# 空间转录组学

## 教程链接

可以参考以下教程开始学习空间转录组分析：

* 空间转录组工具包 [semla](https://ludvigla.github.io/semla/articles/getting_started.html)。除本教程外，本 notebook 中还有两个部分也参考了该包的教程内容，涵盖空间转录组数据的提取和分析。

* [Seurat Spatial Transcriptomics Vignette](https://satijalab.org/seurat/articles/spatial_vignette.html)：如何使用 Seurat 包进行空间转录组分析？


# 安装软件包


In [ ]:
## Function to execute terminal commands in Google Colab with the R kernel
shell_call <- function(command, ...) {
  result <- system(command, intern = TRUE, ...)  # Executes the terminal command and stores the output
  cat(paste0(result, collapse = "\n"))  # Prints the output in a readable format
}
# Downloads the code from the specified URL and saves it as "add_cranapt_jammy.sh"
download.file("https://github.com/eddelbuettel/r2u/raw/master/inst/scripts/add_cranapt_jammy.sh",
              "add_cranapt_jammy.sh")
Sys.chmod("add_cranapt_jammy.sh", "0755")
shell_call("./add_cranapt_jammy.sh")
bspm::enable()
options(bspm.version.check=FALSE)
shell_call("rm add_cranapt_jammy.sh")

# Sets a higher timeout limit to avoid interruptions during package downloads.
options(timeout=1000)

# update the package "vctrs" before install Seurat
install.packages("vctrs", force = TRUE)

In [ ]:
# Install the "semla" package from GitHub, forcing an update.
remotes::install_github("ludvigla/semla", upgrade=T, force = TRUE)

In [ ]:
# Install packages
cranPkgs2Install = c("BiocManager", "ggpubr", "Seurat", "hdf5r", "openxlsx",
                     "enrichR", "clustree", "DT","ggcorrplot","scatterpie", "pheatmap")
install.packages(cranPkgs2Install, ask=FALSE, update=TRUE, quietly=TRUE)

In [ ]:
biocPkgs2Install = c("SingleCellExperiment", "ReactomePA", "org.Hs.eg.db", "glmGamPoi", "fgsea", "limma")
BiocManager::install(biocPkgs2Install, ask=FALSE, update=TRUE, quietly=TRUE)

In [ ]:
# Load packages
# Suppress messages when loading packages to keep the output clean.
suppressPackageStartupMessages({
library(Seurat)  # Framework for single-cell RNA-seq analysis.
library(data.table)  # Efficient manipulation of large datasets.
library(ggplot2)  # Visualization based on the grammar of graphics.
library(plotly)  # Interactive visualizations.
library(RColorBrewer)  # Predefined color palettes.
library(dplyr)  # Data manipulation.
library(semla) # Tools for spatial transcriptomics.
library(clustree)  # Visualization of clustering resolutions.
library(ReactomePA)  # Pathway analysis.
library(org.Hs.eg.db)  # Human gene annotation database.
library(ggpubr)  # Publication-ready visualizations.
library(enrichR)  # Gene enrichment analysis.
library(stringr)  # String manipulation.
library(openxlsx)  # Excel file manipulation.
library(patchwork)  # Combine multiple plots.
library(SingleCellExperiment)  # Structure for single-cell data.
})


# 引言

空间转录组学是一类把基因表达信息与细胞在组织中的物理位置结合起来的技术。不同于传统转录组方法，空间转录组不仅告诉我们哪些基因被表达，还告诉我们这些基因在组织中的什么位置表达，因此可以更精确地研究细胞组织结构和组织微环境。

在这个背景下，组织学图像具有基础性作用。其中最常用的一类是 H&E（hematoxylin and eosin，苏木精-伊红）染色，它通过染色突出组织结构：

* 苏木精将细胞核染成蓝色或紫色。

* 伊红将细胞质和细胞外成分染成粉红色。

把空间转录组数据与 H&E 图像结合起来，可以将分子表达谱与组织结构相互关联，从而更完整地观察细胞在其天然环境中的组织方式和功能状态。


## 下载数据

下面下载的数据来自 10x Genomics 网站，是使用 Space Ranger（https://www.10xgenomics.com/support/software/space-ranger/latest）生成的空间转录组数据集。样本为癌性乳腺组织。

数据详情可见以下链接：

* [Human Breast Cancer (Block A, Section 1)](https://www.10xgenomics.com/datasets/human-breast-cancer-block-a-section-1-1-standard-1-1-0)

* [Human Breast Cancer (Block A, Section 2)](https://www.10xgenomics.com/datasets/human-breast-cancer-block-a-section-2-1-standard-1-1-0)


In [ ]:
shell_call("mkdir -p ST_Exercises")

# Download the files containing spatial transcriptomics exercises
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_2/V1_Breast_Cancer_Block_A_Section_2_filtered_feature_bc_matrix.h5','ST_Exercises/Breast_Cancer_Block_A_Section_2_filtered_feature_bc_matrix.h5')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_2/V1_Breast_Cancer_Block_A_Section_2_spatial.tar.gz','ST_Exercises/Breast_Cancer_Block_A_Section_2_spatial.tar.gz')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_2/V1_Breast_Cancer_Block_A_Section_2_metrics_summary.csv','ST_Exercises/Breast_Cancer_Block_A_Section_2_metrics_summary.csv')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_2/V1_Breast_Cancer_Block_A_Section_2_web_summary.html','ST_Exercises/Breast_Cancer_Block_A_Section_2_web_summary.html')

download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1/V1_Breast_Cancer_Block_A_Section_1_filtered_feature_bc_matrix.h5','ST_Exercises/Breast_Cancer_Block_A_Section_1_filtered_feature_bc_matrix.h5')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1/V1_Breast_Cancer_Block_A_Section_1_spatial.tar.gz','ST_Exercises/Breast_Cancer_Block_A_Section_1_spatial.tar.gz')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1/V1_Breast_Cancer_Block_A_Section_1_metrics_summary.csv','ST_Exercises/Breast_Cancer_Block_A_Section_1_metrics_summary.csv')
download.file('https://cf.10xgenomics.com/samples/spatial-exp/1.1.0/V1_Breast_Cancer_Block_A_Section_1/V1_Breast_Cancer_Block_A_Section_1_web_summary.html','ST_Exercises/Breast_Cancer_Block_A_Section_1_web_summary.html')

# List the files in the current directory with their sizes
shell_call("ls -lh ST_Exercises")


In [ ]:
shell_call("tar -xvzf ST_Exercises/Breast_Cancer_Block_A_Section_1_spatial.tar.gz -C ST_Exercises/")
shell_call("mv ST_Exercises/spatial ST_Exercises/Spatial_Section_1")
shell_call("tar -xvzf ST_Exercises/Breast_Cancer_Block_A_Section_2_spatial.tar.gz -C ST_Exercises/")
shell_call("mv ST_Exercises/spatial ST_Exercises/Spatial_Section_2")

## 加载数据集


In [ ]:
samples <- list.files(
  path = "/content/ST_Exercises",
  pattern = "filtered_feature_bc_matrix.h5",
  full.names = TRUE,
  recursive = TRUE
)

imgs <- list.files(
  path = "/content/ST_Exercises",
  pattern = "tissue_lowres_image.png",
  full.names = TRUE,
  recursive = TRUE
)

spotfiles <- list.files(
  path = "/content/ST_Exercises",
  pattern = "positions_list.csv",
  full.names = TRUE,
  recursive = TRUE
)

json <- list.files(
  path = "/content/ST_Exercises",
  pattern = "scalefactors_json.json",
  full.names = TRUE,
  recursive = TRUE
)

infoTable <- tibble(samples, imgs, spotfiles, json, # Add required columns
                    sample_id = c("Breast_Cancer_Block_A_Section_1", "Breast_Cancer_Block_A_Section_2")) # Add additional column
dim(infoTable)

In [ ]:
# Load matrix
SpatialData <- ReadVisiumData(infoTable)
SpatialData

In [ ]:
# Load a Seurat object.
# load(file = "ST_Exercises/Exercise1_dataset.RData")
# Get dataset information

# This command retrieves the current default assay of the Seurat object SpatialData
DefaultAssay(SpatialData)

# Shows the number of genes (features) and cells.
dim(SpatialData)
head(SpatialData@meta.data)

# This command generates a frequency table of the "sample_id" column 
# in the metadata of the Seurat object SpatialData.
table(SpatialData@meta.data$sample_id)


## 质量控制（QC）图


绘制各项指标。


In [ ]:
# Violin plot with mean values added.
VlnPlot(Her2p, features = "nCount_RNA", group.by = "ids", 
        pt.size = 0.1, cols = color) +
  stat_summary(fun.y=mean, geom="point", shape=95, 
              size=15, color = "black") + NoLegend()

# Scatter plot comparing RNA counts to the number of features (genes).
FeatureScatter(object = Her2p, feature1 = "nCount_RNA", 
feature2 = "nFeature_RNA", group.by = "ids", cols = color)

# Overlays RNA count distribution on spatial coordinates.
FeatureOverlay(Her2p, features = c("nCount_RNA"),
               sampleids = 1:3,
               pt.size = 2.5,
               add.alpha = TRUE,
               ncol = 3, min.cutoff = 0, max.cutoff = 2000)

In [ ]:
# Scatter plot comparing RNA quantity with the number of features (genes)
FeatureScatter(object = SpatialData, feature1 = "nCount_Spatial",
               feature2 = "nFeature_Spatial", group.by = "sample_id", cols = color)


练习：将图像保存到文件。


In [ ]:
# Load H&E images
SpatialData <- LoadImages(SpatialData)
ImagePlot(SpatialData)

In [ ]:
# Mapping tissue cells
MapFeatures(SpatialData, features = "nFeature_Spatial",
            image_use = "raw", override_plot_dims = TRUE) & ThemeLegendRight()

In [ ]:
MapFeaturesSummary(
  object = SpatialData,          # your semla/Seurat object
  features = "nCount_Spatial",   # metric to be displayed
  pt_size = 2.5,                 # point size
  ncol = 2,                      # number of columns in the layout
  subplot_type = "histogram"     # type of plot
)


### 使用 **Seurat** 以另一种方式对空间转录组数据进行图形展示。


In [ ]:
# Create a copy to work with
SpatialData2 <- SpatialData

# Load files into a slot
SpatialData2@images[["slice1"]] <- Read10X_Image(dirname(SpatialData2@tools$Staffli@imgs[1]))
SpatialData2@images[["slice2"]] <- Read10X_Image(dirname(SpatialData2@tools$Staffli@imgs[2]))

# Visualize the data
SpatialFeaturePlot(SpatialData2, features = "nCount_Spatial") + 
                   theme(legend.position = "right")

# Remove the copy
rm(SpatialData2)


归一化是 scRNA-seq 数据分析中非常关键的预处理步骤。它通过调整原始基因表达数据，补偿细胞之间的技术差异和测序深度差异。比较不同归一化方法的结果，有助于理解每种方法的优势和局限。

归一化方法

* SCTransform：一种更高级的方法，使用负二项回归去除技术变异并稳定方差，可为单细胞数据提供更稳健的归一化结果。

* 对数归一化：先缩放基因表达值，再进行对数转换。这有助于稳定方差，使数据更适合后续分析。

去除技术变异

* SCTransform：在去除技术变异方面更稳健，尤其适合变异较大的数据集。

* NormalizeData：同样有效，但在处理技术变异方面不如 SCTransform 精细。

复杂度与运行时间

* SCTransform：由于涉及回归计算和方差稳定化转换，方法更复杂、计算量更大。

* NormalizeData：更简单也更快，适合快速分析和大规模数据集。

协变量处理

* SCTransform：允许把协变量纳入回归模型，例如每个细胞检测到的基因数，从而提升归一化数据质量。

* NormalizeData：通常不会在归一化过程中直接纳入协变量。

方差稳定

* SCTransform：能够稳定方差，使数据更适合后续的聚类和 marker 识别等分析。

* NormalizeData：通过对数转换部分稳定方差，但效果通常不如 SCTransform。

灵活性

* SCTransform：主要面向单细胞 RNA 测序数据，强调稳健归一化。

* NormalizeData：支持不同归一化方法，例如 LogNormalize，可根据具体分析需求调整。

可扩展性

* SCTransform：可以处理大数据集，但由于方法复杂，运行速度可能较慢。

* NormalizeData：得益于方法简单和速度快，能够高效处理大数据集。


In [ ]:
# Create a UMAP plot grouped by "ids", without cluster labels
color = getPalette(9) # Generates a vector of 9 colors using the getPalette function defined earlier.
DimPlot(Her2p, reduction = "umap", label = FALSE, group.by = "ids",
        pt.size = 2, cols=color, label.size=13)

In [ ]:
# Plot only the slide 3 of the patient G
# Overlay the expression of one or more features onto spatial coordinates.
FeatureOverlay(Her2p, # The Seurat object containing your data.
               features = c("SCT_snn_res.0.5"), # Specifies the features to overlay on the spatial coordinates. Clustering resolution 0.5
               sampleids = 2, # Indicates that only the slide with ID 2 should be used for this plot.
               pt.size = 3.5,
               cols = color,
               add.alpha = TRUE, # Add transparency to points
               ncol = 1)

In [ ]:
# Plot only the 3 slide of the patient G
FeatureOverlay(Her2p, features = c("SCT_snn_res.0.4"),
               sampleids = 1:3, # Use slides 1 to 3
               pt.size = 3.5,
               cols = color,
               add.alpha = TRUE,
               ncol = 3) # Arrange plots in 3 columns

In [ ]:
# Normalize data using LogNormalize with a scaling factor
SpatialData <- NormalizeData(object = SpatialData, assay = "Spatial")

In [ ]:
SpatialData <- FindVariableFeatures(SpatialData, nfeatures = 10000)

下面的比较会耗费较长时间，**建议不要运行**。


In [ ]:
# Normalize data using SCTransform
SpatialData <- SCTransform(SpatialData, assay = "Spatial", verbose = TRUE, return.only.var.genes = FALSE)

# Compare normalization methods (LogNormalize vs SCTransform)
SpatialData <- GroupCorrelation(SpatialData, group.assay = "Spatial", assay = "Spatial", layer = "data", do.plot = FALSE)
SpatialData <- GroupCorrelation(SpatialData, group.assay = "Spatial", assay = "SCT", layer = "scale.data", do.plot = FALSE)

# Generate comparison plots
p1 <- GroupCorrelationPlot(SpatialData, assay = "Spatial", cor = "nCount_Spatial_cor") + ggtitle("Log Normalization")
p2 <- GroupCorrelationPlot(SpatialData, assay = "SCT", cor = "nCount_Spatial_cor") + ggtitle("SCTransform Normalization")

# Show both plots side by side
p1 + p2


## 聚类


In [ ]:
# Install package by GitHub
devtools::install_github("zdebruine/RcppML")
devtools::install_github("zdebruine/singlet")

In [ ]:
# Load package
library(singlet)

# Set seed for reproducibility
set.seed(42)
SpatialData2 <- SpatialData

# OPTIONAL: create a subset of data to improve calculation speed
SpatialData2 <- SpatialData2[VariableFeatures(SpatialData2), ]
SpatialData2 <- RunNMF(SpatialData2)


In [ ]:
# Transfer the reductions
SpatialData@reductions  <- SpatialData2@reductions

# Remove the copy
rm(SpatialData2)
gc()

# Transfer the columns
k <- ncol(SpatialData@reductions$nmf@feature.loadings)
k


`RankPlot(SpatialData)` 会生成一张图，展示 NMF 因子按照贡献度排序后的结果，帮助选择最适合解释空间表达模式的因子。


In [ ]:
# RankPlot creates a figure with a ranking data graph.
RankPlot(SpatialData)

我们将观察表达模式在组织空间中的分布方式，这有助于解释细胞异质性。


In [ ]:
# Adjust the dimensions of the plots to be generated
options(repr.plot.width=12, repr.plot.height=6)

# First block of NMF features (from 1 to 6)
MapFeatures(SpatialData,
            features = paste0("NMF_", 1:6),              # Selects NMF_1 to NMF_6 features
            override_plot_dims = TRUE,                   # Forces the dimensions defined above
            colors = viridis::magma(n = 11, direction = -1)) &  # Applies the inverted 'magma' color palette
  theme(plot.title = element_blank())                   # Removes the plot title

# Second block of NMF features (from 7 to 12)
MapFeatures(SpatialData,
            features = paste0("NMF_", 7:12),             # Selects NMF_7 to NMF_12
            override_plot_dims = TRUE,
            colors = viridis::magma(n = 11, direction = -1)) &
  theme(plot.title = element_blank())

# Third block of NMF features (from 13 to 18)
MapFeatures(SpatialData,
            features = paste0("NMF_", 13:18),            # Selects NMF_13 to NMF_18
            override_plot_dims = TRUE,
            colors = viridis::magma(n = 11, direction = -1)) &
  theme(plot.title = element_blank())

# Fourth block of NMF features (from 19 to k)
MapFeatures(SpatialData,
            features = paste0("NMF_", 19:k),             # Selects NMF_19 up to NMF_k (maximum defined value)
            override_plot_dims = TRUE,
            colors = viridis::magma(n = 11, direction = -1)) &
  theme(plot.title = element_blank())


在这一步中，我们基于 NMF 降维结果创建 feature loading plot。这类可视化可以帮助识别哪些基因对前几个潜在成分贡献最大，并把最相关的基因展示在散点图中。这样可以更好地解释空间表达模式的生物学意义，也能理解每个基因如何参与 NMF 捕获到的变异。


In [ ]:
# Generate a feature loadings plot 
# for the spatial data using NMF reduction

PlotFeatureLoadings(SpatialData,
                    dims = 1:2,          # Selects the first two dimensions (components) to visualize
                    reduction = "nmf",   # Specifies that the reduction used is NMF (Non-negative Matrix Factorization)
                    nfeatures = 30,      # Number of most relevant features (genes/variables) to be shown
                    mode = "dotplot",    # Sets the visualization mode as a dot plot
                    fill = "darkmagenta",# Fill color for the points in the plot
                    pt_size = 3)         # Point size in the visualization


在这一步中，我们在空间数据上生成 NMF 的多特征图。该函数可以把所有潜在成分（NMF_1 到 NMF_k）同时投影到原始组织图像上。这里调整了图形尺寸和点大小，以获得清晰且一致的展示效果，便于解释每一种表达模式在组织环境中的空间分布。


In [ ]:
# Adjust the dimensions of the plots to be generated
options(repr.plot.width=16, repr.plot.height=9)

# Generate a multiple-feature map of NMF components in the spatial data
MapMultipleFeatures(SpatialData,
            features = paste0("NMF_", 1:k),# Selects all NMF features from 1 to k
            image_use = "raw",             # Uses the raw image (unprocessed) as the reference background
            override_plot_dims = TRUE,     # Forces the dimensions defined above
            pt_size = 2)                   # Sets the size of the points in the visualization


在这张可视化中，我们希望理解哪些基因主要驱动 NMF 降维得到的各个潜在成分。通过把 feature loadings 绘制为热图，可以清晰比较不同维度上各基因贡献度的变化。


In [ ]:
# Generate a heatmap of feature loadings
# using NMF reduction in the spatial data

PlotFeatureLoadings(SpatialData,
                    dims = 1:k,        # Selects all dimensions from 1 to k
                    reduction = "nmf", # Specifies that the reduction used is NMF (Non-negative Matrix Factorization)
                    nfeatures = 5,     # Shows the 5 most relevant features per dimension
                    mode = "heatmap",  # Sets the visualization mode as heatmap
                    gradient_colors = viridis::magma(n = 11,  # Applies the 'magma' color palette from the viridis package
                                                     direction = -1)) # Inverted to better highlight gradients


In [ ]:
# Plot UMAP
SpatialData <- RunUMAP(SpatialData, reduction = "nmf", dims = 1:k, verbose = FALSE)

In [ ]:
# Perform graph-based clustering with Seurat
# Find the nearest neighbors for each cell using Non-negative Matrix Factorization (NMF)
SpatialData <- FindNeighbors(object = SpatialData,
                      dims = 1:k, # Uses the first k dimensions of the specified reduction method
                      reduction = "nmf", # Indicates that NMF was used as the dimensionality reduction method
                      verbose = FALSE) # Suppresses detailed output

# Define a sequence of resolutions from 0 to 1.2, with increments of 0.2
SpatialData <- FindClusters(SpatialData, resolution = seq(0, 1.2, by = 0.2),
                      verbose = FALSE)


In [ ]:
head(SpatialData@meta.data)

In [ ]:
# Visualize clustering results at different resolutions using Clustree
clustree(SpatialData, prefix = "Spatial_snn_res.")


In [ ]:
# Create a UMAP plot grouped by "sample_id", without cluster labels
DimPlot(SpatialData, reduction = "umap", label = FALSE, group.by = "sample_id",
        pt.size = 2, label.size=13)


In [ ]:
SpatialData2 <- SpatialData
SpatialData2@images[["slice1"]] <- Read10X_Image(dirname(SpatialData2@tools$Staffli@imgs[1]))
SpatialData2@images[["slice2"]] <- Read10X_Image(dirname(SpatialData2@tools$Staffli@imgs[2]))

# Plot only slice 3 of patient G
# Overlay the expression of features on spatial coordinates
SpatialDimPlot(SpatialData2, group.by = "Spatial_snn_res.0.4") + theme(legend.position = "right")

MapMultipleFeatures(SpatialData2,
                    features = paste0("NMF_", 1:k), # Selects all NMF features from 1 to k
                    image_use = "raw",              # Uses the raw tissue image as background
                    override_plot_dims = TRUE,      # Forces the dimensions defined above
                    pt_size = 2)                    # Sets the point size in the visualization


练习：将图像保存到文件。


直接可视化组织空间中的聚类分布，不叠加组织学背景图像。


In [ ]:
# Plot clusters without the histological (HE) image background
SpatialDimPlot(SpatialData2, group.by = "Spatial_snn_res.0.4", image.alpha = 0) + 
  theme(legend.position = "right")   # Visualize clusters defined at resolution 0.4, without image transparency

# Add labels to clusters on the spatial map
MapLabels(SpatialData, column_name = "Spatial_snn_res.0.4", ncol = 2) & 
  theme(legend.position = "right")   # Place the legend on the right for easier reading


## 差异表达基因

在这一部分分析中，我们将在不同空间 cluster 之间识别差异表达 marker 基因（differentially expressed genes, DEGs）。按统计显著性进行过滤，可以确保只保留可信的基因；最后统计每个 cluster 的 DEG 数量，则有助于评估哪些细胞群具有更明确或更富集的分子特征。


In [ ]:
devtools::install_github('immunogenomics/presto')

In [ ]:
DefaultAssay(SpatialData) # Check the active assay of the Seurat SpatialData object
Idents(SpatialData) <- "Spatial_snn_res.0.4" # Set the identity of the object according to clustering resolution 0.4

# Compare all clusters against each other
SpatialData.markers <- FindAllMarkers(object = SpatialData, # Identify differentially expressed genes (DEGs)
                                only.pos = TRUE, # Only genes overexpressed in clusters
                                min.pct = 0.10, # Gene must be expressed in at least 10% of cells
                                logfc.threshold = 0.10) # Minimum log2 fold change threshold

# Filter markers with adjusted p-value < 0.05
SpatialData.markers = SpatialData.markers[which(SpatialData.markers$p_val_adj < 0.05),] # Keep only markers with significant adjusted p-value

# Count the number of DEGs per cluster
table(SpatialData.markers[, "cluster"]) # How many DEGs exist per cluster?


In [ ]:
# Select the top 10 genes with highest logFC per cluster
SpatialData.markers %>% group_by(cluster) %>% top_n(n = 10, wt = avg_log2FC) -> top10

# Generate a dot plot for the selected genes
DotPlot(SpatialData, features = unique(top10$gene),
        group.by = "Spatial_snn_res.0.4", cols = c('#b8d8d8', '#e71d36'),
        dot.scale = 6, col.min = 0) +
        theme(axis.text.x = element_text(face = "bold", color = c("black"),
        size = 8, angle = 90))


练习：将图像保存到文件。


本节展示不同分组之间的表达如何变化，帮助识别哪些细胞群体富集这些基因，从而可以直接比较不同组之间表达强度和表达频率的差异。


In [ ]:
# Plot key genes on a density plot (ridge plot)
RidgePlot(Her2p, assay = "SCT", 
          features = c("IFI27","IFI6"), # Genes de interés
          ncol = 2, group.by = "SCT_snn_res.0.4", 
          cols = color)

In [ ]:
# UMAP visualization and heatmap
# Define the color gradient for the heatmap
heatmap.colors <- c("lightgray", "mistyrose", "red", "darkred", "black")
fts <- c("SPAG6","PGM5-AS1") # List of genes to be visualized

# Generate UMAP plots for each gene
p.fts <- lapply(fts, function(ftr) {
  FeaturePlot(SpatialData, features = ftr, reduction = "umap", order = TRUE, 
              cols = heatmap.colors, pt.size = 2) # Define point size
})

# Plot gene expression mapped to Visium spatial coordinates
p3 <- MapFeatures(SpatialData, features = fts, ncol = 2, cols = heatmap.colors, pt.size = 2)

# Combine all plots into a single figure
cowplot::plot_grid(cowplot::plot_grid(plotlist = p.fts, ncol = 1), p3, ncol = 2, rel_widths = c(1, 1.3))


In [ ]:
# Violin plot for expression of SPAG6 and PGM5-AS1 grouped by clusters
# in the Seurat object 'SpatialData', grouped by clustering resolution "Spatial_snn_res.0.4".
VlnPlot(SpatialData, features = c("SPAG6", "PGM5-AS1"),
        group.by = "Spatial_snn_res.0.4", # Defines the grouping variable (clustering resolution 0.4)
        pt.size = 0, # Sets point size to 0 to avoid plotting individual points
        ncol = 2) +  # Arranges plots in two columns

# Overlay a summary statistic (mean) as a horizontal bar
  stat_summary(fun = mean, geom = "point", shape = 95,
               size = 15, color = "black") +

# Remove the legend from the plot
  NoLegend()


## 3D 可视化


In [ ]:
library(plotly)
library(dplyr)
library(htmlwidgets)

这一步会提取组织图像中每个点或细胞的空间坐标，并按样本进行整理。同时，它还会获取对象关联图像的信息，使表达数据能够与其在组织中的物理位置对应起来。这是空间转录组分析的核心环节，因为它把分子表达谱与组织结构连接起来，并支持在空间上下文中进行生物学解释。


In [ ]:
# Get the spatial coordinates of each spot/cell in the image
xy_coords <- GetCoordinates(SpatialData) |>
    # Note: use pxl_*_in_fullres for the raw image; if sections are aligned,
    # you should use pxl_*_in_fullres_transformed
  dplyr::select(pxl_col_in_fullres, pxl_row_in_fullres, sampleID) |>
  group_by(sampleID) |>
  group_split()   # Split coordinates by sample

# Extract information about the image associated with the spatial object
image_info <- GetImageInfo(SpatialData)


In [ ]:
# Adjust spatial coordinates of each sample
adjusted_coords <- do.call(bind_rows, lapply(seq_along(xy_coords), function(i) {
  xy <- xy_coords[[i]]                        # Extract coordinates of sample i
  full_width <- image_info[i, ]$full_width    # Get the total width of the image
  full_height <- image_info[i, ]$full_height  # Get the total height of the image
  xy <- xy |>
    mutate(x = pxl_col_in_fullres/full_width, # Normalize x coordinate relative to image width
           y = pxl_row_in_fullres/full_height) |> # Normalize y coordinate relative to image height
    # Define a z value using sampleID to separate sections in 3D space.
    # This adjustment allows controlling the distance between aligned sections.
    mutate(z = sampleID*0.2) |>
    dplyr::select(x, y, z)                    # Keep only the adjusted coordinates
}))


In [ ]:
# 3D scatter plot with plotly
p <- plot_ly(adjusted_coords, x = ~x, y = ~y, z = ~z, type = "scatter3d", mode = "markers", color = SpatialData$Spatial_snn_res.0.4, size = 3)
saveWidget(as_widget(p), "SpatialData3DPlot.html")

## Gene Ontology（GO）术语


In [ ]:
# Select differentially expressed genes (DEGs) from cluster 6
geneList = SpatialData.markers[which(SpatialData.markers$cluster == 6), "gene"]
length(geneList)  # Show the number of selected genes

# Convert gene symbols to Entrez IDs
columns(org.Hs.eg.db)  # Show available columns for conversion in the annotation database

symbol <- mapIds(org.Hs.eg.db,
                 keys = geneList,       # List of gene symbols to convert
                 column = "ENTREZID",   # Convert to Entrez IDs
                 keytype = "SYMBOL",    # Input type: gene symbol
                 multiVals = "first")   # If multiple matches, take the first one

symbol = as.vector(symbol)  # Convert to vector format

# Remove genes without Entrez ID
geneList = symbol
geneList = geneList[which(geneList != "NA")]
length(geneList)  # Show the number of genes after filtering

# Pathway enrichment analysis with ReactomePA
x <- enrichPathway(gene = geneList,
                   organism = "human",
                   pvalueCutoff = 0.05,
                   readable = TRUE,
                   pAdjustMethod = "bonferroni")

head(as.data.frame(x))  # Show the first rows of the results

# Order results by adjusted p-value (from smallest to largest)
x@result = x@result[order(x@result$p.adjust, decreasing = FALSE),]

# Select the 30 most significant pathways
x@result = x@result[1:30,]

# Order pathways by gene count (from smallest to largest)
x@result = x@result[order(x@result$Count, decreasing = FALSE),]

# Save the ordered results in a new data frame
newbar.dt = x@result

# Create a bar plot with enriched pathways
cluster6_enrichpathway <- ggbarplot(newbar.dt, x = "Description", y = "Count",
          fill = "Count",         # Color bars according to the number of genes
          color = "white",        # White border color for bars
          sort.val = "asc",       # Sort values in ascending order
          sort.by.groups = FALSE, # Do not sort by group
          x.text.angle = 90,      # Rotate x-axis labels for better visibility
          xlab = "Pathways",
          ylab = "Number of genes",
          legend.title = "Count",
          rotate = TRUE,
          ggtheme = theme_minimal()
) + scale_fill_continuous(low = "#bde0fe", high = "red") +
  theme(text = element_text(size = 20))


## 使用 EnrichR 进行通路分析


In [ ]:
# List available databases in EnrichR
listEnrichrSites()
dbs <- listEnrichrDbs()

# Order databases by name
dbs %>% dplyr::arrange(libraryName)

# Select specific databases for enrichment analysis
dbs <- c("Cancer_Cell_Line_Encyclopedia",
         "Elsevier_Pathway_Collection",
         "KEGG_2021_Human",
         "CellMarker_Augmented_2021",
         "Reactome_2016",
         "GO_Biological_Process_2018",
         "GO_Cellular_Component_2018",
         "GO_Molecular_Function_2018",
         "InterPro_Domains_2019")

# EnrichR requires gene symbols (not Ensembl IDs)
geneList = SpatialData.markers[which(SpatialData.markers$cluster == 6), "gene"]
length(geneList)  # Show the number of selected genes

# Perform enrichment analysis with EnrichR
enriched.Paths <- enrichr(geneList, dbs)

# Show results from selected databases
enriched.Paths[[1]]  # Cancer_Cell_Line_Encyclopedia
enriched.Paths[[2]]  # Elsevier_Pathway_Collection
enriched.Paths[[3]]  # KEGG_2021_Human
enriched.Paths[[4]]  # CellMarker_Augmented_2021
enriched.Paths[[5]]  # Reactome_2016
enriched.Paths[[6]]  # GO_Biological_Process_2018
enriched.Paths[[7]]  # GO_Cellular_Component_2018
enriched.Paths[[8]]  # GO_Molecular_Function_2018
enriched.Paths[[9]]  # InterPro_Domains_2019

# Select results from CellMarker_Augmented_2021
Rpath = enriched.Paths[[4]]

# Count the number of genes per pathway
Gene_Count = c()
for(x in 1:dim(Rpath)[1]){
  Gene_Count = c(Gene_Count, str_split(Rpath$Overlap[x], pattern = "/" )[[1]][1])
}

Rpath$Gene_Count = as.numeric(Gene_Count)

# Order by adjusted p-value (from smallest to largest)
Rpath = Rpath[order(Rpath$Adjusted.P.value, decreasing = FALSE),]

# Filter significant pathways (adjusted p-value < 0.05)
Rpath = Rpath[Rpath$Adjusted.P.value < 0.05,]

# Select the top 20 pathways
Rpath = Rpath[1:20,]

# Order pathways by gene count (from largest to smallest)
Rpath = Rpath[order(Rpath$Gene_Count, decreasing = TRUE),]

# Create a bar plot of enriched pathways
cluster6_enrichR <- ggbarplot(Rpath, x = "Term", y = "Gene_Count",
          fill = "Gene_Count",         # Color bars according to gene count
          color = "white",             # White border color
          sort.val = "asc",            # Sort values in ascending order
          sort.by.groups = FALSE,
          x.text.angle = 90,           # Rotate labels for better visibility
          xlab = "Pathways",
          ylab = "Number of genes",
          legend.title = "Count",
          rotate = TRUE,
          ggtheme = theme_minimal()
) + scale_fill_continuous(low = "#bde0fe", high = "red") +
  theme(text = element_text(size = 18))

# Show the plot
cluster6_enrichR


## 细胞类型去卷积

去卷积是一类计算方法，用于从混合细胞群体中推断并量化不同细胞类型的比例。由于 scRNA-seq 数据通常来自包含多种细胞类型的复杂组织，去卷积可以帮助识别混合样本中各细胞类型的特异性基因表达谱，并估计它们的相对组成。

为什么去卷积重要？

* 细胞类型识别：去卷积可以帮助研究者判断异质性组织样本中存在哪些细胞类型，以及它们的丰度。

* 理解细胞组成：它有助于阐明组织的细胞组成，揭示不同细胞类型如何参与生物过程和疾病进展。

* 改善数据解释：通过分离来自不同细胞类型的基因表达信号，去卷积可以提高 scRNA-seq 数据解释的准确性，使生物学结论更精确。


In [ ]:
download.file('https://github.com/integrativebioinformatics/scNotebooks/blob/main/scNotebooks-Resources/scRNASeq_pac2_processed.rds', 'ST_Exercises/scRNASeq_pac2_processed.rds')

In [ ]:
# Load the scRNA-seq dataset
SC.data = readRDS("ST_Exercises/scRNASeq_pac2_processed.rds")
SC.data <- UpdateSeuratObject(SC.data)

# Set cell identities according to cell type
Idents(SC.data) = "cellType"

# UMAP visualization of cell types
DimPlot(object = SC.data, reduction = 'umap', label = TRUE, label.size = 6,
        group.by = "cellType", pt.size = 1.5)

# Identify DEGs across all clusters
SC.markers <- FindAllMarkers(object = SC.data, only.pos = TRUE, min.pct = 0.10, logfc.threshold = 0.10)
# min.pct = 0.10: at least 10% of cells must express the gene
# logfc.threshold = 0.10: minimum log2 fold change threshold to consider a gene as a marker

SC.markers = SC.markers[which(SC.markers$p_val_adj < 0.05 & SC.markers$avg_log2FC > 0.5),]
# SC.markers$p_val_adj < 0.05: selects markers with adjusted p-value < 0.05 (statistically significant)
# SC.markers$avg_log2FC > 0.5: selects markers with average log2 fold change > 0.5

# Count the number of DEGs per cluster
table(SC.markers$cluster)

DefaultAssay(SC.data) # returns the name of the assay currently set as default in the Seurat object


## 去卷积流程


In [ ]:
#ti <- Sys.time()
DefaultAssay(SpatialData) <- "Spatial"

# Prediction of cell type proportions
SpatialData <- RunNNLS(object = SpatialData,
                      singlecell_object = SC.data,
                      groups = "cellType")

In [ ]:
# Check the available cell types
rownames(SpatialData)

In [ ]:
# Load H&E images
SpatialData <- SpatialData |>
  LoadImages()

In [ ]:
# Plot multiple features
MapMultipleFeatures(SpatialData,
                    image_use = "raw",
                    pt_size = 2, max_cutoff = 0.95,
                    override_plot_dims = TRUE,
                    features = rownames(SpatialData)) +
  plot_layout(guides = "collect")

MapMultipleFeatures(SpatialData,
                    pt_size = 2, max_cutoff = 0.95,
                    override_plot_dims = TRUE,
                    features = rownames(SpatialData)) +
  plot_layout(guides = "collect")

## 细胞类型共定位

细胞类型共定位是指分析不同细胞群体如何在特定组织区域中分布，并是否共同出现。在空间转录组学中，这一步可以识别不同细胞类型之间的邻近或相互作用模式，揭示潜在的功能关系、细胞通讯，或与生物过程和疾病相关的微环境。


In [ ]:
library(pheatmap)

# Extract the expression matrix from SpatialData and compute correlation between genes/cell types
cor_matrix <- FetchData(SpatialData, rownames(SpatialData)) |>
  mutate_all(~ if_else(.x < 0.1, 0, .x)) |>  # Filter very low values, setting them to 0
  cor()                                      # Compute correlation matrix

diag(cor_matrix) <- NA                       # Remove diagonal (self-correlations) for clarity
max_val <- max(cor_matrix, na.rm = T)        # Get maximum correlation value

# Define color palette for heatmap (reversed red-yellow-blue, with white in the center)
cols <- RColorBrewer::brewer.pal(7, "RdYlBu") |> rev(); cols[4] <- "white"

# Adjust figure dimensions
options(repr.plot.width = 6, repr.plot.height = 6)

# Generate heatmap of correlations between cell types within spots
pheatmap::pheatmap(cor_matrix,
                   breaks = seq(-max_val, max_val, length.out = 100), # Correlation value range
                   color = colorRampPalette(cols)(100),              # Color gradient
                   cellwidth = 14, cellheight = 14,                  # Cell size
                   treeheight_col = 10, treeheight_row = 10,         # Dendrogram height
                   main = "Correlation between cell types\nwithin spots") # Plot title


In [ ]:
# Applies non-negative matrix factorization (NMF) to spatial data
nmf_data <- FetchData(SpatialData, rownames(SpatialData)) |>   # Extrai a matriz de expressão do objeto espacial
  RcppML::nmf(k = 10, verbose = T)                             # Executa NMF com k = 10 componentes latentes, mostrando mensagens no console


In [ ]:
# Convert the H matrix from NMF result into a data.frame
nmf_data_h <- nmf_data@h |> as.data.frame()

# Assign row names to factors (Factor_1 to Factor_10)
rownames(nmf_data_h) <- paste0("Factor_", 1:10)

# Assign column names corresponding to cells/spots from the spatial object
colnames(nmf_data_h) <- rownames(SpatialData)

# Normalize values of each column by dividing by the maximum (range between 0 and 1)
nmf_data_h <- nmf_data_h |>
  mutate_at(colnames(nmf_data_h),
            ~(scale(., center = FALSE, scale = max(., na.rm = TRUE)/1)))

# Create a Factor column with row names, keeping the order of factors
nmf_data_h$Factor <- rownames(nmf_data_h) |>
  factor(levels = paste0("Factor_", 1:10))

# Reshape data into long format:
# each row represents the weight of a factor in a specific cell
nmf_data_h_df <- nmf_data_h |>
  tidyr::pivot_longer(cols = all_of(rownames(SpatialData)),
                      names_to = "Cell",
                      values_to = "Weight")


In [ ]:
# Bubble chart
ggplot(nmf_data_h_df, aes(x=Factor, y=Cell, size=Weight, color=Weight)) +
  geom_point() +
  labs(title="Cell type contribution", x="Factor", y = "Cell type",
       color = "", size = "Scaled weight") +
  scale_color_viridis_c(direction = -1, option = "magma") +
  theme_bw() +
  theme(axis.text.x = element_text(angle=45, hjust=1),
        panel.grid = element_blank())